In [ ]:
# Data Preprocessing
# IMport Libraries

# daAta manipulation
import pandas as pd
import numpy as np

# visual and graphs
import matplotlib.pyplot as plt

# ml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# display settings
# pd.set_option("display.max_columns", None)

In [21]:
# Load the Dataset
df = pd.read_csv("../data/customer_churn.csv")

# Display first few rows
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [22]:
# basic overview
print("dataset-shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

dataset-shape: (64374, 12)

Columns:
['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']

Data Types:
CustomerID            int64
Age                   int64
Gender               object
Tenure                int64
Usage Frequency       int64
Support Calls         int64
Payment Delay         int64
Subscription Type    object
Contract Length      object
Total Spend           int64
Last Interaction      int64
Churn                 int64
dtype: object


In [23]:
# checking missing values
missing_values = df.isnull().sum()
print("Missing values:")
print(missing_values)

print("\nTotal Missing Values:", missing_values.sum())

Missing values:
CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

Total Missing Values: 0


In [24]:
# checking duplicates
duplicates = df.duplicated().sum()
print("duplicate rows:", duplicates)

duplicate rows: 0


# to remove customer-if column as it does not help predict customer churn, and may introduce noise.

In [25]:
df.columns

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Churn'],
      dtype='object')

In [26]:
df = df.drop(columns=["CustomerID"])

print(df.head())

   Age  Gender  Tenure  Usage Frequency  Support Calls  Payment Delay  \
0   22  Female      25               14              4             27   
1   41  Female      28               28              7             13   
2   47    Male      27               10              2             29   
3   35    Male       9               12              5             17   
4   53  Female      58               24              9              2   

  Subscription Type Contract Length  Total Spend  Last Interaction  Churn  
0             Basic         Monthly          598                 9      1  
1          Standard         Monthly          584                20      0  
2           Premium          Annual          757                21      0  
3           Premium       Quarterly          232                18      0  
4          Standard          Annual          533                18      0  


In [27]:
df.columns

Index(['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls',
       'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend',
       'Last Interaction', 'Churn'],
      dtype='object')

In [29]:
df["Churn"].value_counts()

Churn
0    33881
1    30493
Name: count, dtype: int64

In [30]:
# check Target variable = 'Churn'
print("Target Distribution:")
print(df["Churn"].value_counts())

print("\nPercentage Distribution:")
print(df["Churn"].value_counts(normalize=True) * 100) # normalise true returns proportions bw 0 and 1
# 0 -> customer stayed
# 1-> customer churned

Target Distribution:
Churn
0    33881
1    30493
Name: count, dtype: int64

Percentage Distribution:
Churn
0    52.631497
1    47.368503
Name: proportion, dtype: float64


In [32]:
df.shape

(64374, 11)

In [31]:
# separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (64374, 10)
Target Shape: (64374,)


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64374 entries, 0 to 64373
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                64374 non-null  int64 
 1   Gender             64374 non-null  object
 2   Tenure             64374 non-null  int64 
 3   Usage Frequency    64374 non-null  int64 
 4   Support Calls      64374 non-null  int64 
 5   Payment Delay      64374 non-null  int64 
 6   Subscription Type  64374 non-null  object
 7   Contract Length    64374 non-null  object
 8   Total Spend        64374 non-null  int64 
 9   Last Interaction   64374 non-null  int64 
 10  Churn              64374 non-null  int64 
dtypes: int64(8), object(3)
memory usage: 5.4+ MB


In [34]:
# numerical or categorical data?
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Subscription Type', 'Contract Length']

Numerical Features:
['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction']


In [ ]:
# pipeline
preprocessor = ColumnTransformer(
    transformers=[(
        "cat",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("num",
         "passthrough",
         numerical_features)]
)

In [38]:
# splitting of thd data
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("training Shape:", X_train.shape)
print("testing Shape :", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True)*100)

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True)*100)

training Shape: (51499, 10)
testing Shape : (12875, 10)

Training Target Distribution:
Churn
0    52.63209
1    47.36791
Name: proportion, dtype: float64

Testing Target Distribution:
Churn
0    52.629126
1    47.370874
Name: proportion, dtype: float64
